# 🏋️ Fine-Tuning de BioMistral-7B com MedQuAD

**Tech Challenge FIAP - Fase 3 | Assistente Médico Inteligente**

---

## 🎯 O que este notebook faz

Pega o modelo **BioMistral-7B** (já pré-treinado em PubMed) e ajusta ele especificamente com perguntas/respostas médicas do dataset **MedQuAD anonimizado**.

É como pegar um **médico recém-formado** (BioMistral) e dar um **curso intensivo** (fine-tuning) sobre os 14.692 Q&As do MedQuAD.

## 📊 Stack técnica

| Componente | Tecnologia | Por quê |
|---|---|---|
| Modelo base | BioMistral-7B | Já viu PubMed, Apache 2.0 |
| Quantização | QLoRA 4-bit | Cabe em A100 (40GB) |
| Framework | Unsloth + PEFT + TRL | 5x mais rápido que LoRA puro |
| GPU | A100 no Colab Pro | 40GB VRAM, 2-4h de treino |
| Dataset | `train.jsonl` (14.692 amostras) | Já curado passo 1+2 |

## ⏱️ Tempo estimado

- Setup: ~5 min
- Download modelo: ~10 min
- Treinamento: ~2-4h (2 epochs)
- Avaliação: ~15 min
- **TOTAL: ~3-5h**

---

# SEÇÃO 1: Setup do ambiente

Antes de tudo, vamos instalar as bibliotecas que vamos usar.

In [ ]:
# ============================================================
# SEÇÃO 1: Setup do ambiente
# ============================================================
# Por que cada linha:
# - torch: deep learning (PyTorch)
# - transformers: carregar BioMistral pré-treinado
# - peft: implementar LoRA (fine-tuning eficiente)
# - bitsandbytes: quantização 4-bit (QLoRA)
# - trl: trainer de fine-tuning supervisionado
# - unsloth: versão otimizada (5x mais rápido)
# - accelerate: gerencia GPU/CPU automaticamente

%%capture  # esconde output detalhado (deixa só o final)
import os
if "COLAB_" not in "".join(os.environ.keys()):
    raise SystemExit("❌ Este notebook deve ser rodado no Google Colab!")

!pip install -q torch==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.0 datasets==2.21.0 peft==0.10.0
!pip install -q bitsandbytes==0.43.3 accelerate==0.33.0 trl==0.10.0
!pip install -q unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git

print("✅ Todas as bibliotecas instaladas com sucesso!")

## 🔍 Verificando GPU

Vamos confirmar que temos uma GPU boa. No Colab Pro, escolha **A100** no menu Runtime > Change runtime type.

In [ ]:
# ============================================================
# SEÇÃO 1.1: Verificar GPU
# ============================================================
import torch

print("=" * 60)
print("🖥️  INFORMAÇÕES DA GPU")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU detectada: {gpu_name}")
    print(f"✅ VRAM disponível: {gpu_memory:.1f} GB")

    # Recomendação
    if gpu_memory >= 35:
        print("🎯 GPU PERFEITA para BioMistral-7B + QLoRA!")
    elif gpu_memory >= 14:
        print("⚠️  GPU OK mas use batch size menor (veja seção 4)")
    else:
        print("❌ GPU muito pequena. Mude para A100 no menu Runtime!")
else:
    print("❌ Nenhuma GPU detectada!")
    print("   Vá em Runtime > Change runtime type > Hardware accelerator > GPU")

# Esperado:
# ✅ GPU detectada: NVIDIA A100-SXM4-40GB
# ✅ VRAM disponível: 40.0 GB
# 🎯 GPU PERFEITA para BioMistral-7B + QLoRA!

---

# SEÇÃO 2: Conectar ao Google Drive

O dataset `train.jsonl` está no seu Drive (ou você pode upar agora). Vamos montar o Drive pra acessar.

In [ ]:
# ============================================================
# SEÇÃO 2: Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Definir pasta de trabalho
# IMPORTANTE: ajuste esse caminho pra onde você salvou os arquivos
WORKDIR = "/content/drive/MyDrive/techchallenge_fase3"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print(f"✅ Working directory: {WORKDIR}")
print(f"📂 Arquivos no diretório:")
!ls -lh {WORKDIR}/

# Esperado:
# Mounted at /content/drive
# ✅ Working directory: /content/drive/MyDrive/techchallenge_fase3
# 📂 Arquivos no diretório:
# -rw-r--r-- 1 root root  17M Aug 31 12:58 train.jsonl
# -rw-r--r-- 1 root root 934K Aug 31 12:58 val.jsonl
# drwx------ 2 root root 4.0K Aug 31 13:00 .config
# etc.

# ⚠️ Se train.jsonl não aparecer, faça upload agora:
# 1. Abra o painel lateral do Drive no Colab
# 2. Navegue até /content/drive/MyDrive/techchallenge_fase3/
# 3. Clique com botão direito > Upload > selecione train.jsonl do seu PC

---

# SEÇÃO 3: Carregar o modelo BioMistral-7B

Aqui é onde a mágica acontece. Vamos baixar o BioMistral-7B (7 bilhões de parâmetros, ~14GB em disco) e configurar ele com QLoRA.

**O que é QLoRA?** Imagine que você tem um livro de 7.000 páginas (o modelo) e quer anotar ele com caneta colorida (fine-tuning). QLoRA te deixa fazer isso sem precisar reescrever o livro — só adiciona 'notas adesivas' que modificam o comportamento.

In [ ]:
# ============================================================
# SEÇÃO 3: Carregar BioMistral-7B com QLoRA
# ============================================================
from unsloth import FastLanguageModel
import torch

# Configurações do modelo
MAX_SEQ_LENGTH = 4096   # tokens máximos por exemplo (instruction + input + output)
DTYPE = None            # auto-detectar (float16 ou bfloat16)
LOAD_IN_4BIT = True     # QLoRA: quantizar pra 4-bit (economiza 4x VRAM)

print("📥 Baixando BioMistral-7B... (pode levar 5-10 min na primeira vez)")

# Carregar modelo base + tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="BioMistral/BioMistral-7B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    # token="hf_XXXXXXXXXX",  # descomente se o modelo exigir autenticação
)

print("\n✅ Modelo BioMistral-7B carregado!")
print(f"   Tipo: {type(model).__name__}")
print(f"   Tokenizer: {type(tokenizer).__name__}")
print(f"   Vocabulário: {tokenizer.vocab_size:,} tokens")

# Esperado após download:
# ✅ Modelo BioMistral-7B carregado!
#    Tipo: FastLanguageModel
#    Tokenizer: LlamaTokenizerFast
#    Vocabulário: 32,000 tokens

## 🎛️ Configurar os adaptadores LoRA

Agora vamos adicionar os 'notas adesivas' (LoRA adapters) ao modelo. Esses são pequenos módulos treináveis que ajustam o comportamento do modelo sem modificar os pesos originais.

In [ ]:
# ============================================================
# SEÇÃO 3.1: Configurar LoRA adapters
# ============================================================
# Parâmetros do LoRA:
# - r=16: rank dos adaptadores (maior = mais capacidade, mais VRAM)
# - target_modules: camadas onde aplicamos LoRA
# - lora_alpha=32: fator de escala (geralmente = 2*r)
# - lora_dropout=0.05: regularização (evita overfitting)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                           # rank LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,                  # fator de escala
    lora_dropout=0.05,              # regularização
    bias="none",                    # não treinar bias
    use_gradient_checkpointing="unsloth",  # economiza VRAM
    random_state=42,                # reprodutibilidade
)

# Verificar quantos parâmetros vamos treinar
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = model.num_parameters()
pct = 100 * trainable_params / all_params

print("=" * 60)
print("📊 PARÂMETROS DO MODELO")
print("=" * 60)
print(f"  Total de parâmetros:    {all_params:>15,} ({all_params/1e9:.2f}B)")
print(f"  Parâmetros treináveis:  {trainable_params:>15,} ({trainable_params/1e6:.2f}M)")
print(f"  Percentual treinável:   {pct:>15.4f}%")
print()
print("💡 Apenas ~0.1% dos parâmetros são treinados!")
print("   É por isso que LoRA é tão rápido e leve.")

# Esperado:
# ============================================================
# 📊 PARÂMETROS DO MODELO
# ============================================================
#   Total de parâmetros:     7,250,000,000 (7.25B)
#   Parâmetros treináveis:      41,000,000 (41.00M)
#   Percentual treinável:            0.5654%
# 
# 💡 Apenas ~0.6% dos parâmetros são treinados!

---

# SEÇÃO 4: Preparar o dataset

Vamos carregar o `train.jsonl` e formatar no template Alpaca (formato padrão de instruction-tuning).

In [ ]:
# ============================================================
# SEÇÃO 4: Carregar e formatar dataset
# ============================================================
from datasets import load_dataset

# Carregar train.jsonl (gerado pelo seu passo 2_normalizar_e_split.py)
dataset = load_dataset("json", data_files="train.jsonl", split="train")
print(f"✅ Dataset carregado: {len(dataset):,} amostras")
print(f"   Colunas: {dataset.column_names}")
print(f"\n📝 Exemplo (amostra 0):")
print(f"   Instruction: {dataset[0]['instruction']}")
print(f"   Input: {dataset[0]['input']}")
print(f"   Output: {dataset[0]['output'][:200]}...")

# Esperado:
# ✅ Dataset carregado: 14,692 amostras
#    Colunas: ['instruction', 'input', 'output']
# 📝 Exemplo (amostra 0):
#    Instruction: What are the treatments for X ?
#    Input: Context / Topic: X
#    Output: ...resposta médica...

In [ ]:
# ============================================================
# SEÇÃO 4.1: Aplicar template Alpaca
# ============================================================
# O template Alpaca padroniza o formato:
# ### Instruction: {instruction}
# ### Input: {input}
# ### Response: {output}

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Token de fim de sequência (importante: sem ele, modelo não sabe parar)
EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    """Formata 1 exemplo no template Alpaca."""
    text = alpaca_prompt.format(
        example["instruction"],
        example.get("input", ""),
        example["output"]
    ) + EOS_TOKEN
    return {"text": text}

# Aplicar formatação em todo dataset
dataset = dataset.map(format_example)

print("=" * 60)
print("✅ Dataset formatado no template Alpaca")
print("=" * 60)
print(f"\n📝 Exemplo formatado (amostra 0):\n")
print(dataset[0]["text"][:800])
print("...")
print(dataset[0]["text"][-100:])

# Esperado:
# ============================================================
# ✅ Dataset formatado no template Alpaca
# ============================================================
#
# 📝 Exemplo formatado (amostra 0):
#
# Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
#
# ### Instruction:
# What are the treatments for Holes in the Heart ?
#
# ### Input:
# Context / Topic: Holes in the Heart
#
# ### Response:
# Many holes in the heart don't need treatment, but some do...</parameter>

---

# SEÇÃO 5: Configurar o Trainer

Aqui configuramos todos os hiperparâmetros do treinamento. Cada um foi escolhido com base em boas práticas pra QLoRA em modelos 7B.

In [ ]:
# ============================================================
# SEÇÃO 5: Configurar SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

# Hiperparâmetros explicados:
# - per_device_train_batch_size=2: quantos exemplos processar por vez (1 GPU)
# - gradient_accumulation_steps=4: acumula gradientes de 4 mini-batches (efetivo batch = 8)
# - num_train_epochs=2: passar pelo dataset inteiro 2 vezes (sweet spot)
# - learning_rate=2e-4: taxa de aprendizado (padrão LoRA)
# - warmup_steps=50: ramp-up inicial pra estabilizar
# - lr_scheduler_type="cosine": decaimento suave do LR
# - optim="adamw_8bit": otimizador com quantização 8-bit (economia VRAM)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,         # batch efetivo = 8
        warmup_steps=50,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,                      # log a cada 20 steps
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./checkpoints",
        save_strategy="epoch",                 # salvar a cada epoch
        save_total_limit=2,                    # manter só 2 últimos
        report_to="none",                      # sem wandb (evita login)
    ),
)

print("✅ Trainer configurado!")
print(f"   Dataset: {len(dataset):,} amostras")
print(f"   Batch size efetivo: 2 × 4 = 8")
print(f"   Steps por epoch: {len(dataset)//8:,}")
print(f"   Total de steps (2 epochs): {len(dataset)//8*2:,}")
print(f"   Tempo estimado em A100: ~2-4 horas")

# Esperado:
# ✅ Trainer configurado!
#    Dataset: 14,692 amostras
#    Batch size efetivo: 2 × 4 = 8
#    Steps por epoch: 1,836
#    Total de steps (2 epochs): 3,672
#    Tempo estimado em A100: ~2-4 horas

---

# SEÇÃO 6: 🚀 TREINAR! (essa é a parte demorada)

Agora vamos **rodar o fine-tuning**. A célula abaixo pode levar 2-4h.

**Dicas:**
- Não feche a aba do Colab
- Pode minimizar a janela — o treino continua rodando
- Você verá mensagens de progresso a cada 20 steps
- Se cair a conexão, o Colab retoma do último checkpoint automaticamente

In [ ]:
# ============================================================
# SEÇÃO 6: TREINAMENTO 🚀
# ============================================================
print("=" * 60)
print("🚀 INICIANDO TREINAMENTO")
print("=" * 60)
print("⏱️  Tempo estimado: 2-4 horas em A100")
print("💡 Pode minimizar essa aba — o treino continua!")
print()

# Iniciar treino (essa linha vai rodar por horas)
trainer.train()

print()
print("=" * 60)
print("✅ TREINAMENTO CONCLUÍDO!")
print("=" * 60)

# Esperado durante o treino (a cada 20 steps):
# {'loss': 1.234, 'grad_norm': 0.5, 'learning_rate': 0.0002, 'epoch': 0.05}
# {'loss': 1.156, 'grad_norm': 0.4, 'learning_rate': 0.000199, 'epoch': 0.06}
# ...
# {'loss': 0.342, 'grad_norm': 0.3, 'learning_rate': 0.00005, 'epoch': 1.99}

# A loss deve cair de ~1.5 pra ~0.3-0.5 em 2 epochs

---

# SEÇÃO 7: Salvar o modelo treinado

Após o treino, salvamos os adaptadores LoRA (são só ~80MB) no Drive pra usar depois.

In [ ]:
# ============================================================
# SEÇÃO 7: Salvar modelo treinado
# ============================================================
OUTPUT_MODEL_DIR = f"{WORKDIR}/biomistral-medquad-lora"

# Salvar adaptadores LoRA + tokenizer
model.save_pretrained(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)

print("=" * 60)
print("💾 MODELO SALVO!")
print("=" * 60)
print(f"📂 Diretório: {OUTPUT_MODEL_DIR}")
print()

# Mostrar arquivos salvos
!ls -lh {OUTPUT_MODEL_DIR}/

import os
size_mb = sum(
    os.path.getsize(os.path.join(OUTPUT_MODEL_DIR, f))
    for f in os.listdir(OUTPUT_MODEL_DIR)
) / 1024 / 1024

print(f"\n📦 Tamanho total: {size_mb:.1f} MB")
print("   (Bem menor que o modelo completo de 14GB!)")

# Esperado:
# ============================================================
# 💾 MODELO SALVO!
# ============================================================
# 📂 Diretório: /content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora
# 
#  -rw-r--r-- 1 root root  17M adapter_model.safetensors
# -rw-r--r-- 1 root root 2.4K adapter_config.json
# -rw-r--r-- 1 root root  130 tokenizer_config.json
# -rw-r--r-- 1 root root 1.1M tokenizer.model
#
# 📦 Tamanho total: 18.1 MB
#    (Bem menor que o modelo completo de 14GB!)

---

# SEÇÃO 8: Testar o modelo com perguntas reais

Vamos ver o modelo em ação fazendo perguntas médicas.

In [ ]:
# ============================================================
# SEÇÃO 8: Inferência — testar o modelo
# ============================================================
from transformers import TextStreamer

# Ativar modo inferência do Unsloth (mais rápido)
FastLanguageModel.for_inference(model)

def perguntar(instruction: str, topic: str = ""):
    """Faz uma pergunta ao modelo fine-tuned."""
    input_text = alpaca_prompt.format(
        instruction,
        f"Context / Topic: {topic}" if topic else "",
        ""  # resposta vazia, modelo vai completar
    )

    inputs = tokenizer([input_text], return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f"❓ PERGUNTA: {instruction}")
    if topic:
        print(f"📋 TÓPICO: {topic}")
    print(f"{'='*70}")
    print(f"🤖 RESPOSTA:")

    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,  # evita repetição
    )
    print()

# Testar com 3 perguntas
perguntar(
    "What are the symptoms of diabetes type 2?",
    "Diabetes Type 2"
)

perguntar(
    "What are the treatments for high blood pressure?",
    "Hypertension"
)

perguntar(
    "Is breast cancer hereditary?",
    "Breast Cancer"
)

# ⚠️ NOTA: as respostas serão em inglês porque o modelo foi treinado em EN
# Para respostas em PT-BR, você precisaria adicionar dados sintéticos em português
# (próximo passo do projeto)

---

# SEÇÃO 9: Avaliação quantitativa no test set

Agora vamos medir a perplexity (métrica clássica de qualidade de LLM) no conjunto de validação.

In [ ]:
# ============================================================
# SEÇÃO 9: Avaliação — perplexity no val set
# ============================================================
import math
from datasets import load_dataset

val_dataset = load_dataset("json", data_files="val.jsonl", split="val")
print(f"📂 Carregado val set: {len(val_dataset):,} amostras\n")

total_loss = 0
n_samples = 0

# Limitar a 100 amostras pra evaluation rápida (perplexity não precisa de dataset inteiro)
MAX_EVAL_SAMPLES = 100

model.eval()  # modo avaliação (sem dropout)
with torch.no_grad():
    for i, example in enumerate(val_dataset):
        if i >= MAX_EVAL_SAMPLES:
            break

        text = alpaca_prompt.format(
            example["instruction"],
            example.get("input", ""),
            example["output"]
        )
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to("cuda")

        outputs = model(**inputs, labels=inputs["input_ids"])
        total_loss += outputs.loss.item()
        n_samples += 1

        if (i + 1) % 20 == 0:
            print(f"   Avaliado {i+1}/{MAX_EVAL_SAMPLES} amostras...")

perplexity = math.exp(total_loss / n_samples)

print()
print("=" * 60)
print("📊 RESULTADO DA AVALIAÇÃO")
print("=" * 60)
print(f"  Loss média:    {total_loss / n_samples:.4f}")
print(f"  Perplexity:    {perplexity:.2f}")
print()
print("💡 Interpretação da perplexity:")
print(f"   • < 5   = Sobreajuste (modelo 'decorou' o val set)"")
print(f"   • 5-15 = Excelente (modelo aprendeu o domínio)"")
print(f"   • 15-30 = Bom"")
print(f"   • > 50 = Modelo ainda não aprendeu bem"")

# Esperado para BioMistral fine-tuned em MedQuAD:
# Loss média: 0.85
# Perplexity: 2.34
# (Perplexity baixa pq o modelo 'aprendeu bem' o domínio)

---

# SEÇÃO 10: Avaliação qualitativa — gerar respostas para 20 perguntas do test set

Vou gerar respostas para 20 perguntas aleatórias do test set e salvar pra você revisar manualmente.

In [ ]:
# ============================================================
# SEÇÃO 10: Gerar respostas para revisão qualitativa
# ============================================================
import json
import random

test_dataset = load_dataset("json", data_files="test.jsonl", split="test")

# Pegar 20 amostras aleatórias
rng = random.Random(42)
indices = rng.sample(range(len(test_dataset)), 20)
samples = [test_dataset[i] for i in indices]

results = []
print(f"🚀 Gerando respostas para {len(samples)} perguntas...\n")

for i, example in enumerate(samples):
    input_text = alpaca_prompt.format(
        example["instruction"],
        example.get("input", ""),
        ""
    )
    inputs = tokenizer([input_text], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("### Response:")[-1].strip()

    results.append({
        "instruction": example["instruction"],
        "expected": example["output"][:500],
        "generated": response[:500],
    })

    print(f"  [{i+1}/20] {example['instruction'][:60]}... OK")

# Salvar resultados
output_eval = f"{WORKDIR}/eval_results_qualitativo.json"
os.makedirs(os.path.dirname(output_eval), exist_ok=True)
with open(output_eval, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Resultados salvos em: {output_eval}")
print(f"   {len(results)} pares (esperado vs gerado) para revisão manual")

# Mostrar 3 exemplos lado a lado
print("\n" + "=" * 70)
print("📋 EXEMPLOS LADO A LADO (esperado vs gerado)")
print("=" * 70)
for r in results[:3]:
    print(f"\n--- PERGUNTA ---")
    print(f"   {r['instruction']}")
    print(f"\n--- ESPERADO (gabarito) ---")
    print(f"   {r['expected'][:300]}...")
    print(f"\n--- GERADO PELO MODELO ---")
    print(f"   {r['generated'][:300]}...")
    print()

---

# SEÇÃO 11: Upload para HuggingFace Hub (opcional)

Se quiser versionar seu modelo no HuggingFace Hub pra compartilhar (privado ou público), use esta célula.

**⚠️ Você precisa de um token de acesso**: https://huggingface.co/settings/tokens

In [ ]:
# ============================================================
# SEÇÃO 11: Upload para HuggingFace Hub (OPCIONAL)
# ============================================================
# Descomente as linhas abaixo e adicione seu token HF

"""
from huggingface_hub import HfApi, notebook_login

# Login no HF
notebook_login()  # vai pedir seu token

# Nome do repo (mude para seu username)
HF_REPO = "seu-username/biomistral-medquad-lora"  # ← MUDE AQUI

# Upload
model.push_to_hub(HF_REPO, private=True)  # private=True = só você vê
tokenizer.push_to_hub(HF_REPO, private=True)

print(f"✅ Modelo enviado para: https://huggingface.co/{HF_REPO}")
"""

print("ℹ️  Esta célula está comentada. Descomente para usar.")
print("   Você precisa de uma conta no HuggingFace + criar um token.")

---

# 🎉 CONCLUSÃO

Parabéns! Você completou o **fine-tuning do BioMistral-7B** com o dataset MedQuAD.

## 📁 O que você tem agora

1. **Modelo fine-tuned**: `biomistral-medquad-lora/` (~80MB) — pronto pra usar no pipeline LangChain
2. **Resultados qualitativos**: `eval_results_qualitativo.json` — 20 pares pra revisar
3. **Checkpoints**: `./checkpoints/` — versões intermediárias
4. **Relatórios do treino**: loss/perplexity impressos

## 🚀 Próximos passos do Tech Challenge

| # | Etapa | O que fazer |
|---|---|---|
| 1 | ✅ Fine-tuning | **VOCÊ ESTÁ AQUI** |
| 2 | ⏭️ RAG #1 PMC | Indexar PubMed Central (próximo notebook) |
| 3 | ⏭️ RAG #2 Interno | Indexar base interna do hospital |
| 4 | ⏭️ Agentes LangGraph | Implementar 3 agentes (Triagem, Síntese, Validação) |
| 5 | ⏭️ Guardrails + HITL | Validação humana obrigatória |
| 6 | ⏭️ Geração de documentos | Prontuário, receita, atestado (PDF) |
| 7 | ⏭️ UI Gradio | Interface para o médico |
| 8 | ⏭️ Vídeo demo | Gravar ≤15min demonstrando tudo |

## 📚 Documentação adicional

- **DOCX gigante**: `TECHCHALLENGE_FASE3_PROJETO_COMPLETO.docx`
- **README**: estrutura completa do projeto
- **Notebook RAG**: `notebooks/03_rag_pmc.ipynb` (próximo a criar)

---

**Desenvolvido para**: Tech Challenge FIAP - Fase 3
**Autor**: Michelle Nogueira
**Data**: 2026-08-31